# Foundation models & generalist agents — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
print('setup ok')

## Scaling laws and routing in a generalist model
The core concept is one broad pretrained prior used across tasks. These examples visualize the scaling curve, compute multiplier effects, rough training compute, mixture task loss, and tool-routing probabilities.

In [ ]:
# 1) Scaling loss falls smoothly with compute
A=1.2; alpha=0.08; Cs=np.logspace(20,24,80); L=A*Cs**(-alpha)
plt.figure(figsize=(6,3)); plt.loglog(Cs,L); plt.scatter([1e20,1e22,1e24],[A*(1e20)**(-alpha),A*(1e22)**(-alpha),A*(1e24)**(-alpha)],c='r')
plt.xlabel('compute C'); plt.ylabel('loss'); plt.title('L(C)=1.2*C^-0.08')
assert abs(A*(1e20)**(-alpha)-0.030142637178114957)<1e-15

In [ ]:
# 2) A 100x compute increase multiplies loss by 100^-alpha
mults=np.logspace(0,3,80); factors=mults**(-0.08); factor=100**(-0.08)
plt.figure(figsize=(6,3)); plt.semilogx(mults,factors); plt.scatter([100],[factor],c='r'); plt.title(f'100x compute -> loss factor {factor:.3f}')
plt.xlabel('compute multiplier'); plt.ylabel('loss multiplier')
assert abs(factor-0.6918309709189365)<1e-15

In [ ]:
# 3) Rough transformer compute C ≈ 6ND
N=70e9; Ds=np.array([50e9,100e9,300e9,600e9]); C=6*N*Ds
plt.figure(figsize=(6,3)); plt.plot(Ds/1e9,C,'-o'); plt.scatter([300],[6*N*300e9],c='r'); plt.xlabel('training tokens (billions)'); plt.ylabel('FLOPs'); plt.title('compute grows linearly with tokens')
assert abs(6*N*300e9-1.26e23)<1e10

In [ ]:
# 4) A generalist evaluation is a weighted mixture
losses=np.array([0.20,0.40,0.60]); weights=np.array([0.5,0.3,0.2]); mix=float(weights@losses)
plt.figure(figsize=(6,3)); plt.bar(['coding','dialogue','tool use'],losses); plt.axhline(mix,ls='--',c='r',label='weighted average'); plt.legend(); plt.title(f'mixture loss = {mix:.3f}')
assert abs(mix-0.34)<1e-12

In [ ]:
# 5) Tool routing uses softmax probabilities and expected cost
logits=np.array([2.0,1.0,-0.5]); p=np.exp(logits)/np.exp(logits).sum(); costs=np.array([0.2,0.5,0.8]); expected=float(p@costs)
plt.figure(figsize=(6,3)); plt.bar(['answer','retrieve','calculate'],p); plt.title(f'expected tool cost = {expected:.3f}')
plt.ylabel('routing probability')
assert np.allclose(p,[0.68967209,0.25371618,0.05661173],atol=1e-8) and abs(expected-0.3100818938347903)<1e-12